In [ ]:
import json,pickle,zipfile,re,math,os,time,copy
import numpy as np
import unicodedata
import torch, torch.nn as nn, torch.nn.functional as F
from collections import Counter
from tqdm import tqdm
from scipy import sparse
from tqdm.auto import tqdm
from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient
from torch.utils.data import DataLoader
from torch.utils import checkpoint
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

In [ ]:
## tokenizer
 
WORD_END = 256
 
def pre_tokenize_preserve(text):
    return re.findall(r"\w+|[^\w\s]|\s+", text)
 
def word_to_symbols(word):
    byte_values = word.encode("utf-8")
    symbols = [(b,) for b in byte_values]
    symbols.append((WORD_END,))
    return tuple(symbols)
 
def get_adjacent_pairs(symbols):
    return set(zip(symbols, symbols[1:]))
 
def apply_bpe_merges(symbols, merge_rank):
    symbols = list(symbols)
    while len(symbols) > 1:
        pairs = get_adjacent_pairs(symbols)
        candidates = [p for p in pairs if p in merge_rank]
        if not candidates:
            break
        best_pair = min(candidates, key=lambda p: merge_rank[p])
        sa, sb = best_pair
        merged = sa + sb
        new_symbols, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == sa and symbols[i + 1] == sb:
                new_symbols.append(merged)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        symbols = new_symbols
    return symbols
 
def load_tokenizer(pkl_path):
    bpe_data = pickle.load(open(pkl_path, "rb"))
    stoi, merge_rank = bpe_data["stoi"], bpe_data["merge_rank"]
    cache = {}
 
    def encode_unit(unit):
        if unit not in cache:
            cache[unit] = [stoi[s] for s in apply_bpe_merges(word_to_symbols(unit), merge_rank)]
        return cache[unit]
 
    def encode(text):
        ids = []
        for unit in pre_tokenize_preserve(text):
            ids += encode_unit(" " if unit.isspace() else unit)
        return ids
    return bpe_data, encode
 
def clean_text(t):
    t = unicodedata.normalize("NFKC", t)
    t = re.sub(r"(\w)-[ \t]*\n[ \t]*(\w)", r"\1\2", t)
    return re.sub(r"\s+", " ", t).strip()

# MODEL V2

In [ ]:
class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len, base=10000.0):
        super().__init__()
        assert head_dim % 2 == 0, "RoPE requires an even head_dim"
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self.max_seq_len = max_seq_len
        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len):
        t = torch.arange(seq_len, dtype=self.inv_freq.dtype, device=self.inv_freq.device)
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)

    def forward(self, seq_len, offset, device, dtype):
        needed = offset + seq_len
        if needed > self.cos_cached.shape[0]:
            self._build_cache(needed)
        cos = self.cos_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        sin = self.sin_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        return cos, sin


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rope(x, cos, sin):
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return (x * cos) + (rotate_half(x) * sin)

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, context_length, dropout=0.1):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.n_embd = n_embd
        self.head_dim = n_embd // n_head
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.attn_dropout_p = dropout
        self.resid_dropout = nn.Dropout(dropout)
        self.context_length = context_length
        self.rope = RotaryEmbedding(self.head_dim, context_length)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False, position_offset=0):
        B, T, C = x.shape
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        cos, sin = self.rope(T, position_offset, x.device, x.dtype)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
        present = (k, v) if use_cache else None

        dropout_p = self.attn_dropout_p if self.training else 0.0
        if attn_mask is not None:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, dropout_p=dropout_p)
        elif past_kv is not None:
            y = F.scaled_dot_product_attention(q, k, v, dropout_p=dropout_p)
        else:
            y = F.scaled_dot_product_attention(q, k, v, dropout_p=dropout_p, is_causal=True)

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), present


class MLP(nn.Module):
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class Block(nn.Module):
    def __init__(self, n_embd, n_head, context_length, dropout=0.1):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, context_length, dropout)
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False, position_offset=0):
        attn_out, present = self.attn(self.ln_1(x), attn_mask, past_kv, use_cache, position_offset)
        x = x + attn_out
        x = x + self.mlp(self.ln_2(x))
        return (x, present) if use_cache else x

class DecoderOnlyLM_RoPE(nn.Module):
    def __init__(self, vocab_size, context_length, n_embd, n_layer, n_head,
                dropout=0.1, pad_id=0, gradient_checkpointing=False):
        super().__init__()
        self.vocab_size = vocab_size
        self.context_length = context_length
        self.pad_id = pad_id
        self.gradient_checkpointing = gradient_checkpointing
        self.wte = nn.Embedding(vocab_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            Block(n_embd, n_head, context_length, dropout) for _ in range(n_layer)
        ])
        self.ln_f = nn.LayerNorm(n_embd)

        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("c_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.LayerNorm):
            nn.init.zeros_(module.bias)
            nn.init.ones_(module.weight)

    def forward(self, input_ids, attention_mask=None, labels=None,
                past_key_values=None, use_cache=False, last_only=False):
        B, T = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)
        assert past_len + T <= self.context_length, \
            f"seq len {past_len + T} > context_length {self.context_length}"

        combined_mask = None
        if attention_mask is not None:
            key_mask = attention_mask.bool()[:, None, None, :]
            if past_len == 0:
                causal = torch.tril(torch.ones(T, T, dtype=torch.bool, device=input_ids.device))
                eye = torch.eye(T, dtype=torch.bool, device=input_ids.device)
                combined_mask = (causal[None, None] & key_mask) | eye[None, None]
            else:
                combined_mask = key_mask

        x = self.drop(self.wte(input_ids))

        presents = [] if use_cache else None
        for i, block in enumerate(self.blocks):
            past = past_key_values[i] if past_key_values is not None else None
            if self.gradient_checkpointing and self.training and not use_cache:
                x = checkpoint.checkpoint(block, x, combined_mask, None, False, past_len,
                                          use_reentrant=False)
            else:
                out = block(x, combined_mask, past, use_cache, past_len)
                if use_cache:
                    x, present = out
                    presents.append(present)
                else:
                    x = out

        x = self.ln_f(x)
        if last_only:
            x = x[:, -1:, :]
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=self.pad_id,
            )
        return {"loss": loss, "logits": logits, "past_key_values": presents}

    def _sample_next_token(self, logits, temperature, top_p, top_k):
        logits = logits / max(temperature, 1e-5)
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float("inf")
        probs = F.softmax(logits, dim=-1)
        if top_p is not None:
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cumulative = torch.cumsum(sorted_probs, dim=-1)
            sorted_mask = cumulative - sorted_probs > top_p
            sorted_probs[sorted_mask] = 0.0
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_token = torch.multinomial(sorted_probs, num_samples=1)
            return torch.gather(sorted_idx, -1, next_token)
        return torch.multinomial(probs, num_samples=1)

    @torch.no_grad()
    def generate_batch(self, input_ids, attention_mask, max_new_tokens,
                        temperature=0.7, top_p=0.95, top_k=50, eos_id=None):
        self.eval()
        B, T0 = input_ids.shape
        assert T0 + max_new_tokens <= self.context_length, "prompt + new tokens exceed context_length"
        finished = torch.zeros(B, dtype=torch.bool, device=input_ids.device)

        out = self.forward(input_ids, attention_mask=attention_mask, use_cache=True, last_only=True)
        past, logits = out["past_key_values"], out["logits"][:, -1, :]

        for step in range(max_new_tokens):
            next_token = self._sample_next_token(logits, temperature, top_p, top_k)

            if eos_id is not None:
                next_token = torch.where(
                    finished.unsqueeze(1), torch.full_like(next_token, eos_id), next_token
                )

            input_ids = torch.cat([input_ids, next_token], dim=1)
            attention_mask = torch.cat([attention_mask, torch.ones_like(next_token)], dim=1)

            if eos_id is not None:
                finished = finished | (next_token.squeeze(1) == eos_id)
                if finished.all():
                    break
            if step == max_new_tokens - 1:
                break

            out = self.forward(next_token, attention_mask=attention_mask,
                               past_key_values=past, use_cache=True, last_only=True)
            past, logits = out["past_key_values"], out["logits"][:, -1, :]

        return input_ids


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

# MODEL V5

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm_x = x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return norm_x * self.weight

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len, base=10000.0):
        super().__init__()
        assert head_dim % 2 == 0
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len):
        t = torch.arange(seq_len, dtype=self.inv_freq.dtype, device=self.inv_freq.device)
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)

    def forward(self, seq_len, offset, device, dtype):
        needed = offset + seq_len
        if needed > self.cos_cached.shape[0]:
            self._build_cache(needed)
        cos = self.cos_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        sin = self.sin_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        return cos, sin


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(x, cos, sin):
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return (x * cos) + (rotate_half(x) * sin)


class GroupedQueryAttention(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, context_length, dropout=0.1):
        super().__init__()
        assert n_embd % n_head == 0
        assert n_head % n_kv_head == 0
        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.n_rep = n_head // n_kv_head
        self.head_dim = n_embd // n_head

        self.q_proj = nn.Linear(n_embd, n_head * self.head_dim, bias=False)
        self.k_proj = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.v_proj = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=False)

        self.attn_dropout_p = dropout
        self.resid_dropout = nn.Dropout(dropout)
        self.rope = RotaryEmbedding(self.head_dim, context_length)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False, position_offset=0):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)

        cos, sin = self.rope(T, position_offset, x.device, x.dtype)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
        present = (k, v) if use_cache else None

        k_exp = k.repeat_interleave(self.n_rep, dim=1)
        v_exp = v.repeat_interleave(self.n_rep, dim=1)

        dropout_p = self.attn_dropout_p if self.training else 0.0
        if attn_mask is not None:
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, attn_mask=attn_mask, dropout_p=dropout_p)
        elif past_kv is not None:
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, dropout_p=dropout_p)
        else:
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, dropout_p=dropout_p, is_causal=True)

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), present

class Expert(nn.Module):
    def __init__(self, n_embd, expert_hidden_mult=4, dropout=0.1):
        super().__init__()
        hidden = expert_hidden_mult * n_embd
        self.c_fc = nn.Linear(n_embd, hidden)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(hidden, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class MoEFeedForward(nn.Module):
    def __init__(self, n_embd, num_experts=4, expert_hidden_mult=4, dropout=0.1):
        super().__init__()
        self.num_experts = num_experts
        self.router = nn.Linear(n_embd, num_experts, bias=False)
        self.experts = nn.ModuleList([
            Expert(n_embd, expert_hidden_mult, dropout) for _ in range(num_experts)
        ])

    def forward(self, x):
        B, T, C = x.shape
        x_flat = x.view(-1, C)                                  

        router_logits = self.router(x_flat)                     
        router_probs = F.softmax(router_logits, dim=-1)
        top1_prob, top1_idx = router_probs.max(dim=-1)          
        one_hot = F.one_hot(top1_idx, num_classes=self.num_experts).to(x.dtype)   

        expert_outputs = torch.stack([expert(x_flat) for expert in self.experts], dim=1) 
        gate = (one_hot * top1_prob.unsqueeze(-1)).unsqueeze(-1)                          
        output = (expert_outputs * gate).sum(dim=1)                                        

        with torch.no_grad():
            tokens_per_expert = one_hot.float().mean(dim=0)     
        mean_router_prob = router_probs.mean(dim=0)              
        aux_loss = self.num_experts * (tokens_per_expert * mean_router_prob).sum()

        return output.view(B, T, C), aux_loss

class Block(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, context_length, num_experts, dropout=0.1):
        super().__init__()
        self.ln_1_pre = RMSNorm(n_embd)
        self.attn = GroupedQueryAttention(n_embd, n_head, n_kv_head, context_length, dropout)
        self.ln_1_post = RMSNorm(n_embd)

        self.ln_2_pre = RMSNorm(n_embd)
        self.moe = MoEFeedForward(n_embd, num_experts=num_experts, dropout=dropout)
        self.ln_2_post = RMSNorm(n_embd)

    def forward_train(self, x, attn_mask):
        attn_out, _ = self.attn(self.ln_1_pre(x), attn_mask)
        x = x + self.ln_1_post(attn_out)

        moe_out, aux_loss = self.moe(self.ln_2_pre(x))
        x = x + self.ln_2_post(moe_out)
        return x, aux_loss

    def forward_infer(self, x, past_kv, use_cache, position_offset):
        attn_out, present = self.attn(self.ln_1_pre(x), None, past_kv, use_cache, position_offset)
        x = x + self.ln_1_post(attn_out)

        moe_out, _ = self.moe(self.ln_2_pre(x))
        x = x + self.ln_2_post(moe_out)
        return x, present



class DecoderOnlyLM_MoE_Sandwich(nn.Module):
    def __init__(self, vocab_size, context_length, n_embd, n_layer, n_head, n_kv_head,
                num_experts=4, aux_loss_weight=0.01, dropout=0.1, pad_id=0,
                gradient_checkpointing=False):
        super().__init__()
        self.vocab_size = vocab_size
        self.context_length = context_length
        self.pad_id = pad_id
        self.aux_loss_weight = aux_loss_weight
        self.gradient_checkpointing = gradient_checkpointing

        self.wte = nn.Embedding(vocab_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            Block(n_embd, n_head, n_kv_head, context_length, num_experts, dropout)
            for _ in range(n_layer)
        ])
        self.ln_f = RMSNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("c_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, attention_mask=None, labels=None,
                past_key_values=None, use_cache=False, last_only=False):
        B, T = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)
        assert past_len + T <= self.context_length, \
            f"seq len {past_len + T} > context_length {self.context_length}"

        x = self.drop(self.wte(input_ids))

        if use_cache:
            presents = []
            for i, block in enumerate(self.blocks):
                past = past_key_values[i] if past_key_values is not None else None
                x, present = block.forward_infer(x, past, use_cache, past_len)
                presents.append(present)
        else:
            combined_mask = None
            if attention_mask is not None:
                key_mask = attention_mask.bool()[:, None, None, :]
                causal = torch.tril(torch.ones(T, T, dtype=torch.bool, device=input_ids.device))
                eye = torch.eye(T, dtype=torch.bool, device=input_ids.device)
                combined_mask = (causal[None, None] & key_mask) | eye[None, None]

            aux_losses = []
            for block in self.blocks:
                if self.gradient_checkpointing and self.training:
                    x, aux = checkpoint.checkpoint(block.forward_train, x, combined_mask,
                                                   use_reentrant=False)
                else:
                    x, aux = block.forward_train(x, combined_mask)
                aux_losses.append(aux)
            presents = None

        x = self.ln_f(x)
        if last_only:
            x = x[:, -1:, :]
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            ce_loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=self.pad_id,
            )
            aux_loss = torch.stack(aux_losses).mean() if not use_cache else torch.tensor(0.0, device=x.device)
            loss = ce_loss + self.aux_loss_weight * aux_loss

        return {"loss": loss, "logits": logits, "past_key_values": presents}

    def _sample_next_token(self, logits, temperature, top_p, top_k):
        logits = logits / max(temperature, 1e-5)
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float("inf")
        probs = F.softmax(logits, dim=-1)
        if top_p is not None:
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cumulative = torch.cumsum(sorted_probs, dim=-1)
            sorted_mask = cumulative - sorted_probs > top_p
            sorted_probs[sorted_mask] = 0.0
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_token = torch.multinomial(sorted_probs, num_samples=1)
            return torch.gather(sorted_idx, -1, next_token)
        return torch.multinomial(probs, num_samples=1)

    @torch.no_grad()
    def generate_batch(self, input_ids, attention_mask, max_new_tokens,
                        temperature=0.7, top_p=0.95, top_k=50, eos_id=None):
        self.eval()
        B, T0 = input_ids.shape
        assert T0 + max_new_tokens <= self.context_length, "prompt + new tokens exceed context_length"
        finished = torch.zeros(B, dtype=torch.bool, device=input_ids.device)

        out = self.forward(input_ids, attention_mask=attention_mask, use_cache=True, last_only=True)
        past, logits = out["past_key_values"], out["logits"][:, -1, :]

        for step in range(max_new_tokens):
            next_token = self._sample_next_token(logits, temperature, top_p, top_k)

            if eos_id is not None:
                next_token = torch.where(
                    finished.unsqueeze(1), torch.full_like(next_token, eos_id), next_token
                )

            input_ids = torch.cat([input_ids, next_token], dim=1)

            if eos_id is not None:
                finished = finished | (next_token.squeeze(1) == eos_id)
                if finished.all():
                    break
            if step == max_new_tokens - 1:
                break

            out = self.forward(next_token, attention_mask=None,
                               past_key_values=past, use_cache=True, last_only=True)
            past, logits = out["past_key_values"], out["logits"][:, -1, :]

        return input_ids


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


def count_active_parameters(model, num_experts):
    total = count_parameters(model)
    expert_params = sum(p.numel() for p in model.blocks[0].moe.experts[0].parameters())
    num_layers = len(model.blocks)
    unused_experts_params = expert_params * (num_experts - 1) * num_layers
    return total - unused_experts_params

# MODEL V6

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm_x = x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return norm_x * self.weight


class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len, base=10000.0):
        super().__init__()
        assert head_dim % 2 == 0
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len):
        t = torch.arange(seq_len, dtype=self.inv_freq.dtype, device=self.inv_freq.device)
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)

    def forward(self, seq_len, offset, device, dtype):
        needed = offset + seq_len
        if needed > self.cos_cached.shape[0]:
            self._build_cache(needed)
        cos = self.cos_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        sin = self.sin_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        return cos, sin


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(x, cos, sin):
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return (x * cos) + (rotate_half(x) * sin)


def is_global_layer(layer_idx, n_layer, global_every_n):
    return (n_layer - 1 - layer_idx) % global_every_n == 0

class GroupedQueryAttention(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, context_length, window_size=None, dropout=0.1):
        super().__init__()
        assert n_embd % n_head == 0
        assert n_head % n_kv_head == 0
        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.n_rep = n_head // n_kv_head
        self.head_dim = n_embd // n_head
        self.window_size = window_size  

        self.q_proj = nn.Linear(n_embd, n_head * self.head_dim, bias=False)
        self.k_proj = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.v_proj = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=False)

        self.attn_dropout_p = dropout
        self.resid_dropout = nn.Dropout(dropout)
        self.rope = RotaryEmbedding(self.head_dim, context_length)

    def _build_mask(self, T, device, key_padding_mask=None):
        i = torch.arange(T, device=device)
        if self.window_size is None:
            causal = i[:, None] >= i[None, :]                          
        else:
            diff = i[:, None] - i[None, :]
            causal = (diff >= 0) & (diff < self.window_size)              
        mask = causal[None, None, :, :]
        if key_padding_mask is not None:
            key_mask = key_padding_mask.bool()[:, None, None, :]
            eye = torch.eye(T, dtype=torch.bool, device=device)
            mask = (mask & key_mask) | eye[None, None, :, :]
        return mask

    def forward(self, x, key_padding_mask=None, past_kv=None, use_cache=False, position_offset=0):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)

        cos, sin = self.rope(T, position_offset, x.device, x.dtype)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
            if self.window_size is not None and k.size(2) > self.window_size:
                k = k[:, :, -self.window_size:, :]
                v = v[:, :, -self.window_size:, :]
                if key_padding_mask is not None:
                    key_padding_mask = key_padding_mask[:, -self.window_size:]
        present = (k, v) if use_cache else None

        k_exp = k.repeat_interleave(self.n_rep, dim=1)
        v_exp = v.repeat_interleave(self.n_rep, dim=1)

        dropout_p = self.attn_dropout_p if self.training else 0.0
        if past_kv is None and T > 1:
            attn_mask = self._build_mask(T, x.device, key_padding_mask)
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, attn_mask=attn_mask, dropout_p=dropout_p)
        else:
            attn_mask = None
            if key_padding_mask is not None:
                attn_mask = key_padding_mask.bool()[:, None, None, :]
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, attn_mask=attn_mask, dropout_p=dropout_p)

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), present


class MLP(nn.Module):
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class Block(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, context_length, window_size, dropout=0.1):
        super().__init__()
        self.ln_1_pre = RMSNorm(n_embd)
        self.attn = GroupedQueryAttention(n_embd, n_head, n_kv_head, context_length, window_size, dropout)
        self.ln_1_post = RMSNorm(n_embd)
        self.ln_2_pre = RMSNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)
        self.ln_2_post = RMSNorm(n_embd)

    def forward(self, x, key_padding_mask=None, past_kv=None, use_cache=False, position_offset=0):
        attn_out, present = self.attn(self.ln_1_pre(x), key_padding_mask, past_kv, use_cache, position_offset)
        x = x + self.ln_1_post(attn_out)
        x = x + self.ln_2_post(self.mlp(self.ln_2_pre(x)))
        return (x, present) if use_cache else x

class DecoderOnlyLM_HybridMask(nn.Module):
    def __init__(self, vocab_size, context_length, n_embd, n_layer, n_head, n_kv_head,
                window_size=192, global_every_n=4, dropout=0.1, pad_id=0,
                gradient_checkpointing=False):
        super().__init__()
        self.vocab_size = vocab_size
        self.context_length = context_length
        self.pad_id = pad_id
        self.gradient_checkpointing = gradient_checkpointing
        self.window_size = window_size
        self.global_every_n = global_every_n

        self.wte = nn.Embedding(vocab_size, n_embd)
        self.drop = nn.Dropout(dropout)

        layer_windows = [
            None if is_global_layer(i, n_layer, global_every_n) else window_size
            for i in range(n_layer)
        ]
        print(f"Layer attention pattern (None = global/full-causal): {layer_windows}")

        self.blocks = nn.ModuleList([
            Block(n_embd, n_head, n_kv_head, context_length, layer_windows[i], dropout)
            for i in range(n_layer)
        ])
        self.ln_f = RMSNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("c_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, attention_mask=None, labels=None,
                past_key_values=None, use_cache=False, last_only=False):
        B, T = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)
        assert past_len + T <= self.context_length, \
            f"seq len {past_len + T} > context_length {self.context_length}"

        if attention_mask is None and past_key_values is None:
            attention_mask = input_ids.ne(self.pad_id)

        x = self.drop(self.wte(input_ids))

        presents = [] if use_cache else None
        for i, block in enumerate(self.blocks):
            past = past_key_values[i] if past_key_values is not None else None
            if self.gradient_checkpointing and self.training and not use_cache:
                x = checkpoint.checkpoint(block, x, attention_mask, None, False, past_len,
                                          use_reentrant=False)
            else:
                out = block(x, attention_mask, past, use_cache, past_len)
                if use_cache:
                    x, present = out
                    presents.append(present)
                else:
                    x = out

        x = self.ln_f(x)
        if last_only:
            x = x[:, -1:, :]
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=self.pad_id,
            )
        return {"loss": loss, "logits": logits, "past_key_values": presents}

    def _sample_next_token(self, logits, temperature, top_p, top_k):
        logits = logits / max(temperature, 1e-5)
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float("inf")
        probs = F.softmax(logits, dim=-1)
        if top_p is not None:
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cumulative = torch.cumsum(sorted_probs, dim=-1)
            sorted_mask = cumulative - sorted_probs > top_p
            sorted_probs[sorted_mask] = 0.0
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_token = torch.multinomial(sorted_probs, num_samples=1)
            return torch.gather(sorted_idx, -1, next_token)
        return torch.multinomial(probs, num_samples=1)

    @torch.no_grad()
    def generate_batch(self, input_ids, attention_mask, max_new_tokens,
                        temperature=0.7, top_p=0.95, top_k=50, eos_id=None):
        self.eval()
        B, T0 = input_ids.shape
        assert T0 + max_new_tokens <= self.context_length, "prompt + new tokens exceed context_length"
        finished = torch.zeros(B, dtype=torch.bool, device=input_ids.device)

        out = self.forward(input_ids, attention_mask=attention_mask, use_cache=True, last_only=True)
        past, logits = out["past_key_values"], out["logits"][:, -1, :]
        running_mask = attention_mask

        for step in range(max_new_tokens):
            next_token = self._sample_next_token(logits, temperature, top_p, top_k)

            if eos_id is not None:
                next_token = torch.where(
                    finished.unsqueeze(1), torch.full_like(next_token, eos_id), next_token
                )

            input_ids = torch.cat([input_ids, next_token], dim=1)
            running_mask = torch.cat([running_mask, torch.ones_like(next_token)], dim=1)

            if eos_id is not None:
                finished = finished | (next_token.squeeze(1) == eos_id)
                if finished.all():
                    break
            if step == max_new_tokens - 1:
                break

            out = self.forward(next_token, attention_mask=running_mask,
                               past_key_values=past, use_cache=True, last_only=True)
            past, logits = out["past_key_values"], out["logits"][:, -1, :]

        return input_ids


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

# MODEL V7

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm_x = x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return norm_x * self.weight


class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len, base=10000.0):
        super().__init__()
        assert head_dim % 2 == 0, "RoPE requires an even head_dim"
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self.max_seq_len = max_seq_len
        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len):
        t = torch.arange(seq_len, dtype=self.inv_freq.dtype, device=self.inv_freq.device)
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)

    def forward(self, seq_len, offset, device, dtype):
        needed = offset + seq_len
        if needed > self.cos_cached.shape[0]:
            self._build_cache(needed)
        cos = self.cos_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        sin = self.sin_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        return cos, sin


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(x, cos, sin):
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return (x * cos) + (rotate_half(x) * sin)

class GroupedQueryAttention(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, context_length, dropout=0.1):
        super().__init__()
        assert n_embd % n_head == 0
        assert n_head % n_kv_head == 0, "n_head must be divisible by n_kv_head"
        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.n_rep = n_head // n_kv_head
        self.head_dim = n_embd // n_head

        self.q_proj = nn.Linear(n_embd, n_head * self.head_dim, bias=False)
        self.k_proj = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.v_proj = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=False)

        self.attn_dropout_p = dropout
        self.resid_dropout = nn.Dropout(dropout)
        self.context_length = context_length
        self.rope = RotaryEmbedding(self.head_dim, context_length)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False, position_offset=0):
        B, T, C = x.shape

        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)

        cos, sin = self.rope(T, position_offset, x.device, x.dtype)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
        present = (k, v) if use_cache else None
        k_exp = k.repeat_interleave(self.n_rep, dim=1)
        v_exp = v.repeat_interleave(self.n_rep, dim=1)

        dropout_p = self.attn_dropout_p if self.training else 0.0
        if attn_mask is not None:
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, attn_mask=attn_mask, dropout_p=dropout_p)
        elif past_kv is not None:
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, dropout_p=dropout_p)
        else:
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, dropout_p=dropout_p, is_causal=True)

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), present


class MLP(nn.Module):
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class Block(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, context_length, dropout=0.1):
        super().__init__()
        self.ln_1 = RMSNorm(n_embd)
        self.attn = GroupedQueryAttention(n_embd, n_head, n_kv_head, context_length, dropout)
        self.ln_2 = RMSNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False, position_offset=0):
        attn_out, present = self.attn(self.ln_1(x), attn_mask, past_kv, use_cache, position_offset)
        x = x + attn_out
        x = x + self.mlp(self.ln_2(x))
        return (x, present) if use_cache else x

class DecoderOnlyLM_RMSNorm_GQA(nn.Module):
    def __init__(self, vocab_size, context_length, n_embd, n_layer, n_head, n_kv_head,
                dropout=0.1, pad_id=0, gradient_checkpointing=False):
        super().__init__()
        self.vocab_size = vocab_size
        self.context_length = context_length
        self.pad_id = pad_id
        self.gradient_checkpointing = gradient_checkpointing
        self.wte = nn.Embedding(vocab_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            Block(n_embd, n_head, n_kv_head, context_length, dropout) for _ in range(n_layer)
        ])
        self.ln_f = RMSNorm(n_embd)

        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("c_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, attention_mask=None, labels=None,
                past_key_values=None, use_cache=False, last_only=False):
        B, T = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)
        assert past_len + T <= self.context_length, \
            f"seq len {past_len + T} > context_length {self.context_length}"

        combined_mask = None
        if attention_mask is not None:
            key_mask = attention_mask.bool()[:, None, None, :]
            if past_len == 0:
                causal = torch.tril(torch.ones(T, T, dtype=torch.bool, device=input_ids.device))
                eye = torch.eye(T, dtype=torch.bool, device=input_ids.device)
                combined_mask = (causal[None, None] & key_mask) | eye[None, None]
            else:
                combined_mask = key_mask

        x = self.drop(self.wte(input_ids))

        presents = [] if use_cache else None
        for i, block in enumerate(self.blocks):
            past = past_key_values[i] if past_key_values is not None else None
            if self.gradient_checkpointing and self.training and not use_cache:
                x = checkpoint.checkpoint(block, x, combined_mask, None, False, past_len,
                                          use_reentrant=False)
            else:
                out = block(x, combined_mask, past, use_cache, past_len)
                if use_cache:
                    x, present = out
                    presents.append(present)
                else:
                    x = out

        x = self.ln_f(x)
        if last_only:
            x = x[:, -1:, :]
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=self.pad_id,
            )
        return {"loss": loss, "logits": logits, "past_key_values": presents}

    def _sample_next_token(self, logits, temperature, top_p, top_k):
        logits = logits / max(temperature, 1e-5)
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float("inf")
        probs = F.softmax(logits, dim=-1)
        if top_p is not None:
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cumulative = torch.cumsum(sorted_probs, dim=-1)
            sorted_mask = cumulative - sorted_probs > top_p
            sorted_probs[sorted_mask] = 0.0
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_token = torch.multinomial(sorted_probs, num_samples=1)
            return torch.gather(sorted_idx, -1, next_token)
        return torch.multinomial(probs, num_samples=1)

    @torch.no_grad()
    def generate_batch(self, input_ids, attention_mask, max_new_tokens,
                        temperature=0.7, top_p=0.95, top_k=50, eos_id=None):
        self.eval()
        B, T0 = input_ids.shape
        assert T0 + max_new_tokens <= self.context_length, "prompt + new tokens exceed context_length"
        finished = torch.zeros(B, dtype=torch.bool, device=input_ids.device)

        out = self.forward(input_ids, attention_mask=attention_mask, use_cache=True, last_only=True)
        past, logits = out["past_key_values"], out["logits"][:, -1, :]

        for step in range(max_new_tokens):
            next_token = self._sample_next_token(logits, temperature, top_p, top_k)

            if eos_id is not None:
                next_token = torch.where(
                    finished.unsqueeze(1), torch.full_like(next_token, eos_id), next_token
                )

            input_ids = torch.cat([input_ids, next_token], dim=1)
            attention_mask = torch.cat([attention_mask, torch.ones_like(next_token)], dim=1)

            if eos_id is not None:
                finished = finished | (next_token.squeeze(1) == eos_id)
                if finished.all():
                    break
            if step == max_new_tokens - 1:
                break

            out = self.forward(next_token, attention_mask=attention_mask,
                               past_key_values=past, use_cache=True, last_only=True)
            past, logits = out["past_key_values"], out["logits"][:, -1, :]

        return input_ids


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


In [ ]:
def build_model():
    # for  v7
    return DecoderOnlyLM_RMSNorm_GQA(
    vocab_size=VOCAB_SIZE,          
    context_length=1536,
    n_embd=768, 
    n_layer=10,
    n_head=12,  
    n_kv_head=3,
    pad_id=PAD_ID,                  
    gradient_checkpointing=True,
) 
# for v2
#       return DecoderOnlyLM_RoPE(
#     vocab_size=VOCAB_SIZE,          
#     context_length=1536,
#     n_embd=640,
#     n_layer=10,
#     n_head=10,
#     pad_id=PAD_ID,                  
#     gradient_checkpointing=True,
# )
# for v5
#       return DecoderOnlyLM_MoE_Sandwich(
#     vocab_size=VOCAB_SIZE,          
#     context_length=1536,
#     n_embd=640,
#     n_layer=10,
#     n_head=10,
#     n_kv_head=2,
#     num_experts=2,
#     aux_loss_weight=0.01,
#     pad_id=PAD_ID,                  
#     gradient_checkpointing=True,
# )
# for v6
#     return DecoderOnlyLM_HybridMask(
#     vocab_size=VOCAB_SIZE,          
#     context_length=1536,
#     n_embd=640,
#     n_layer=10,
#     n_head=10,
#     n_kv_head=2,
#     window_size=192,
#     global_every_n=4,
#     pad_id=PAD_ID,                  
#     gradient_checkpointing=True,
# )


HEAD_PATH   = "lm_head"     
LAYERS_PATH = "blocks"           
FINAL_NORM  = ("ln_f",)     
MODEL_PT    = "/kaggle/working/decoder_baseline_v7_final_model.pt"

In [ ]:
def load_phase1(build_fn, path):
    model = build_fn()
    ckpt = torch.load(path, map_location="cpu")
    sd = ckpt["model_state"] if isinstance(ckpt, dict) and "model_state" in ckpt else ckpt
    model.load_state_dict({k.replace("module.", "", 1): v for k, v in sd.items()}, strict=True)
    return model

def get_logits(out):
    return out["logits"] if isinstance(out, dict) else out

def count_params(model):
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in model.parameters())
    print(f"trainable {tr/1e6:.3f}M / {tot/1e6:.2f}M ({100*tr/max(1,tot):.2f}%)")
    return tr, tot

def trainable_state(model):
    return {n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}

TEXT_KEYS = ("text", "document", "judgment", "doc")
REL_KEYS = ("relevant_candidates", "relevant", "labels")

def _join(x):
    return " ".join(str(s) for s in x) if isinstance(x, list) else str(x)

def _records(raw):
    if isinstance(raw, dict):
        if isinstance(raw.get("data"), list):
            return raw["data"]
        return [({"id": k, **v} if isinstance(v, dict) else {"id": k, "text": v}) for k, v in raw.items()]
    return raw

def normalize_cases(raw):
    out = []
    for d in _records(raw):
        txt = next((d[k] for k in TEXT_KEYS if k in d), None)
        if txt is None:
            raise ValueError(f"case {d.get('id')!r} has no text field (tried {TEXT_KEYS}); keys: {list(d)}")
        cid = d.get("id", d.get("case_id", d.get("query_id")))
        rel = next((d[k] for k in REL_KEYS if k in d), [])
        out.append({"id": str(cid), "text": _join(txt), "rel": [str(r) for r in rel]})
    return out

def pcr_paths(root, include_test=False):
    p = {"train": (f"{root}/Train/train.json", f"{root}/Train/train_candidates.json"),
         "dev": (f"{root}/Dev/dev.json", f"{root}/Dev/dev_candidates.json")}
    if include_test:
        p["test"] = (f"{root}/Test/test.json", f"{root}/Test/test_candidates.json")
    return p

def download_pcr_files(repo="Exploration-Lab/CS-781-Capstone", p7="Phase_7_PCR", token=None, include_test=False):
    return {s: tuple(hf_hub_download(repo_id=repo, filename=f, repo_type="dataset", token=token) for f in fs) for s, fs in pcr_paths(p7, include_test).items()}

def tokenize_cases(items, encode_fn, clean_fn, max_doc_tokens, desc):
    out = []
    for it in tqdm(items, desc=desc, leave=False):
        ids = np.asarray(encode_fn(clean_fn(it["text"]) if clean_fn else it["text"]), dtype=np.int32)
        if max_doc_tokens and len(ids) > max_doc_tokens:
            ids = ids[:max_doc_tokens]
        out.append({"id": it["id"], "ids": ids, "rel": it["rel"]})
    return out

def load_split(name, files, encode, clean_fn, max_doc_tokens=None, cache=None):
    key = {"clean": bool(clean_fn), "max_doc_tokens": max_doc_tokens}
    data = None
    if cache and os.path.exists(cache):
        c = pickle.load(open(cache, "rb"))
        if c.get("key") == key:
            data = c
            print(f"[{name}] loaded tokenized split from {cache}")
    if data is None:
        queries = tokenize_cases(normalize_cases(json.load(open(files[0], encoding="utf-8"))), encode, clean_fn, max_doc_tokens, f"{name} queries")
        pool = tokenize_cases(normalize_cases(json.load(open(files[1], encoding="utf-8"))), encode, clean_fn, max_doc_tokens, f"{name} candidates")
        data = {"key": key, "queries": queries, "pool": pool}
        if cache:
            pickle.dump(data, open(cache, "wb"))
            print(f"[{name}] saved tokenized split to {cache}")
    queries, pool = data["queries"], data["pool"]
    pidx = {p["id"]: i for i, p in enumerate(pool)}
    gold, miss = [], 0
    for q in queries:
        g = [pidx[r] for r in q["rel"] if r in pidx]
        miss += len(q["rel"]) - len(g)
        gold.append(np.array(g, dtype=np.int64))
    ql = np.array([len(q["ids"]) for q in queries]); pl = np.array([len(p["ids"]) for p in pool])
    print(f"[{name}] {len(queries)} queries / {len(pool)} candidates | gold per query "
          f"{np.mean([len(q['rel']) for q in queries]):.2f} (gold ids missing from pool: {miss}) | "
          f"tokens: query median {int(np.median(ql))} p90 {int(np.percentile(ql, 90))}, candidate median {int(np.median(pl))} p90 {int(np.percentile(pl, 90))}")
    return {"name": name, "queries": queries, "pool": pool, "pidx": pidx, "gold": gold,"self_idx": [pidx.get(q["id"], -1) for q in queries]}

def setup_pcr(paths, tokenizer_pkl, splits=("train", "dev"), eos_token="</s>", clean=True, clean_fn=None,max_doc_tokens=None, cache_prefix=None):
    bpe_data, encode = load_tokenizer(tokenizer_pkl)
    stoi = bpe_data["stoi"]
    clean_fn = (clean_fn or clean_text) if clean else None
    D = {"pad_id": stoi["<pad>"], "end_id": stoi.get(eos_token), "vocab_size": bpe_data["vocab_size"], "encode": encode,"clean": clean_fn, "max_doc_tokens": max_doc_tokens}
    for s in splits:
        D[s] = load_split(s, paths[s], encode, clean_fn, max_doc_tokens, f"{cache_prefix}_{s}.pkl" if cache_prefix else None)
    print(f"PAD_ID={D['pad_id']} END_ID={D['end_id']} VOCAB={D['vocab_size']} cleaning={'on' if clean_fn else 'off'}")
    return D

def add_split(D, paths, name, cache_prefix=None):
    D[name] = load_split(name, paths[name], D["encode"], D["clean"], D["max_doc_tokens"],f"{cache_prefix}_{name}.pkl" if cache_prefix else None)
    return D[name]

def _feats(ids, V, bigrams, nb=2 ** 18):
    ids = ids.astype(np.int64)
    if bigrams and len(ids) > 1:
        ids = np.concatenate([ids, V + (ids[:-1] * 1000003 + ids[1:]) % nb])
    return ids

In [ ]:
class Lexical:
    def __init__(self, pool, V, bigrams=True):
        self.V, self.bigrams = V, bigrams
        self.dim = V + (2 ** 18 if bigrams else 0)
        X = self._counts(pool)
        df = np.asarray((X > 0).sum(0)).ravel()
        self.idf = (np.log((len(pool) + 1) / (df + 1)) + 1).astype(np.float32)
        self.X = self._weight(X)

    def _counts(self, docs):
        indptr, indices, data = [0], [], []
        for d in docs:
            u, c = np.unique(_feats(d["ids"], self.V, self.bigrams), return_counts=True)
            indices.append(u); data.append(c); indptr.append(indptr[-1] + len(u))
        return sparse.csr_matrix((np.concatenate(data).astype(np.float32), np.concatenate(indices), np.array(indptr)),shape=(len(docs), self.dim))

    def _weight(self, X):
        X = X.copy(); X.data = 1 + np.log(X.data)
        X = (X @ sparse.diags(self.idf)).tocsr()
        norm = np.sqrt(np.asarray(X.multiply(X).sum(1)).ravel())
        return (sparse.diags(1 / np.maximum(norm, 1e-9)) @ X).tocsr()

    def scores(self, queries):
        Q = self._weight(self._counts(queries))
        return (Q @ self.X.T).toarray().astype(np.float32)

def lexical_scores(split, D, bigrams=True):
    if "_lex" not in split:
        split["_lex"] = Lexical(split["pool"], D["vocab_size"], bigrams).scores(split["queries"])
    return split["_lex"]

def mine_hard_negatives(split, D, top=50, skip=0):
    S = lexical_scores(split, D)
    hn = []
    for qi in range(len(S)):
        s = S[qi].copy()
        s[split["gold"][qi]] = -np.inf
        if split["self_idx"][qi] >= 0:
            s[split["self_idx"][qi]] = -np.inf
        c = np.argpartition(-s, top + skip)[:top + skip]
        hn.append(c[np.argsort(-s[c])][skip:])
    split["hard_neg"] = hn
    print(f"[{split['name']}] mined {top} hard negatives (skip {skip}) for {len(hn)} queries")

def rank_top(scores, split, k=7):
    s = scores.astype(np.float64).copy()
    for qi, si in enumerate(split["self_idx"]):
        if si >= 0:
            s[qi, si] = -np.inf
    top = np.argpartition(-s, k, axis=1)[:, :k]
    order = np.argsort(-np.take_along_axis(s, top, 1), axis=1)
    return np.take_along_axis(top, order, 1)

def score_f1(split, scores, k=7):
    top = rank_top(scores, split, k)
    tp = sum(len(set(top[i].tolist()) & set(g.tolist())) for i, g in enumerate(split["gold"]))
    n_gold = sum(len(q["rel"]) for q in split["queries"])
    P, R = tp / (k * len(top)), tp / max(1, n_gold)
    return {"f1": 2 * P * R / (P + R) if P + R else 0.0, "precision": P, "recall": R}

def zrow(S):
    return (S - S.mean(1, keepdims=True)) / (S.std(1, keepdims=True) + 1e-8)

def tune_hybrid(split, dense, lex, ws=(0, 0.1, 0.2, 0.3, 0.5, 0.75, 1, 1.5, 2, 4)):
    rows = []
    for w in ws:
        m = score_f1(split, zrow(dense) + w * zrow(lex))
        rows.append((m["f1"], w, m))
        print(f"w={w:<5} F1@7 {m['f1']:.4f}  P {m['precision']:.4f}  R {m['recall']:.4f}")
    best = max(rows, key=lambda r: r[0])
    print("BEST w =", best[1], "F1@7 =", round(best[0], 4))
    return best[1]

def _with_end(ids, end_id):
    return (np.append(ids, end_id) if end_id is not None else ids).astype(np.int64)

def select_window(ids, n, mode, end_id, head_frac=0.5):
    m = n - (1 if end_id is not None else 0)
    if len(ids) > m:
        if mode == "head":
            ids = ids[:m]
        elif mode == "headtail":
            h = int(m * head_frac); ids = np.concatenate([ids[:h], ids[len(ids) - (m - h):]])
        elif mode == "spread":
            w = m // 4; st = np.linspace(0, len(ids) - w, 4).round().astype(int)
            ids = np.concatenate([ids[s:s + w] for s in st])
        else:
            raise ValueError(f"unknown window mode {mode!r}")
    return _with_end(ids, end_id)

def eval_windows(ids, n, mode, end_id, windows):
    m = n - (1 if end_id is not None else 0)
    if windows <= 1 or len(ids) <= m:
        return [select_window(ids, n, mode, end_id)]
    return [_with_end(ids[s:s + m], end_id) for s in np.linspace(0, len(ids) - m, windows).round().astype(int)]

def to_encoder(model, head_path="lm_head"):
    parent_path, _, child = head_path.rpartition(".")
    setattr(model.get_submodule(parent_path) if parent_path else model, child, nn.Identity())
    return model

def pool_hidden(h, lens, pool):
    B, T, _ = h.shape
    h = h.float()
    if pool == "mean":
        mask = (torch.arange(T, device=h.device)[None] < lens[:, None]).unsqueeze(-1)
        e = (h * mask).sum(1) / lens[:, None].float()
    elif pool == "last":
        e = h[torch.arange(B, device=h.device), lens - 1]
    else:
        raise ValueError(pool)
    return F.normalize(e, dim=-1)

def embed_seqs(model, seqs, pool, pad_id, device, amp):
    L = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), L), pad_id, dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = torch.from_numpy(s)
    lens = torch.tensor([len(s) for s in seqs])
    ids, lens = ids.to(device), lens.to(device)
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
        h = get_logits(model(ids))
    return pool_hidden(h, lens, pool)

@torch.no_grad()
def encode_docs(model, docs, cfg, D, device, desc="encoding"):
    model.eval()
    amp = cfg["fp16"] and device.type == "cuda"
    items = [(i, w) for i, d in enumerate(docs)
             for w in eval_windows(d["ids"], cfg["eval_tokens"], cfg["window_mode"], D["end_id"], cfg["eval_windows"])]
    order = sorted(range(len(items)), key=lambda j: len(items[j][1]))
    acc, bs = None, cfg["eval_batch"]
    for s in tqdm(range(0, len(order), bs), desc=desc, leave=False):
        idx = order[s:s + bs]
        e = embed_seqs(model, [items[j][1] for j in idx], cfg["pool"], D["pad_id"], device, amp).cpu()
        if acc is None:
            acc = torch.zeros(len(docs), e.size(1))
        acc.index_add_(0, torch.tensor([items[j][0] for j in idx]), e)
    return F.normalize(acc, dim=-1)

def evaluate_pcr(model, split, cfg, D, device, k=7, verbose=True):
    q = encode_docs(model, split["queries"], cfg, D, device, "queries")
    p = encode_docs(model, split["pool"], cfg, D, device, "candidates")
    scores = (q @ p.T).numpy()
    m = score_f1(split, scores, k)
    if verbose:
        print(f"[{split['name']}] F1@{k} {m['f1']:.4f} | P {m['precision']:.4f} | R {m['recall']:.4f}")
    return m, scores

def contrastive_loss(q_e, d_e, tau, gold_sets, doc_idx):
    B = q_e.size(0)
    sc = (q_e @ d_e.T).float() / tau
    mask = torch.zeros_like(sc, dtype=torch.bool)
    for i in range(B):
        for k, j in enumerate(doc_idx):
            if k != i and j in gold_sets[i]:
                mask[i, k] = True
    return F.cross_entropy(sc.masked_fill(mask, -1e4), torch.arange(B, device=sc.device))

def train_pcr(model, tr, dev, D, cfg, device, save_path=None, extra_loss=None, on_step=None, resume=False):
    model.to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    B, H = cfg["batch_queries"], cfg["n_hard"]
    valid = [qi for qi, g in enumerate(tr["gold"]) if len(g) > 0]
    spe = len(valid) // B
    total = max(1, spe * cfg["epochs"]); warm = int(cfg["warmup_ratio"] * total)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: s / max(1, warm) if s < warm else max(0.0, (total - s) / max(1, total - warm)))
    amp = cfg["fp16"] and device.type == "cuda"
    scaler = torch.amp.GradScaler(enabled=amp)
    best, hist, step, start = -1.0, [], 0, 0
    last_path = save_path.replace("_best.pt", "_last.pt") if save_path else None
    if resume and last_path and os.path.exists(last_path):
        st = torch.load(last_path, map_location=device)
        model.load_state_dict(st["model"], strict=False)
        opt.load_state_dict(st["opt"]); sched.load_state_dict(st["sched"]); scaler.load_state_dict(st["scaler"])
        start, best, hist, step = st["epoch"], st["best"], st["history"], st["step"]
        print(f"resumed from {last_path}: {start} epoch(s) done, best dev F1 {best:.4f}")
    print(f"{len(valid)} usable train queries ({len(tr['gold']) - len(valid)} without gold in pool), {spe} steps/epoch,"f"{B} queries + {B} positives + {B * H} hard negatives per step")
    for epoch in range(start, cfg["epochs"]):
        rng = np.random.default_rng(cfg["seed"] + epoch)
        order = rng.permutation(valid)
        model.train()
        run, t0 = 0.0, time.time()
        pbar = tqdm(range(spe), desc=f"epoch {epoch + 1}/{cfg['epochs']}", leave=False)
        for s in pbar:
            qis = order[s * B:(s + 1) * B]
            win = lambda ids: select_window(ids, cfg["train_tokens"], cfg["window_mode"], D["end_id"])
            q_seqs = [win(tr["queries"][qi]["ids"]) for qi in qis]
            doc_idx = [int(rng.choice(tr["gold"][qi])) for qi in qis]
            for qi in qis:
                doc_idx += [int(j) for j in rng.choice(tr["hard_neg"][qi], H, replace=False)]
            d_seqs = [win(tr["pool"][j]["ids"]) for j in doc_idx]
            emb = embed_seqs(model, q_seqs + d_seqs, cfg["pool"], D["pad_id"], device, amp)
            loss = contrastive_loss(emb[:B], emb[B:], cfg["tau"], [set(tr["gold"][qi].tolist()) for qi in qis], doc_idx)
            if extra_loss is not None:
                loss = loss + extra_loss(model)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, cfg["max_grad_norm"])
            scaler.step(opt); scaler.update(); sched.step(); step += 1
            if on_step is not None:
                on_step(model, step)
            run += loss.item()
            pbar.set_postfix(loss=f"{run / (s + 1):.4f}")
        rec = {"epoch": epoch + 1, "train_loss": run / max(1, spe), "sec": round(time.time() - t0)}
        m, _ = evaluate_pcr(model, dev, cfg, D, device, verbose=False)
        rec.update(val=m["f1"], precision=m["precision"], recall=m["recall"])
        if m["f1"] > best:
            best = m["f1"]
            if save_path:
                torch.save(trainable_state(model), save_path)
        hist.append(rec); print(rec)
        if last_path:
            tmp = last_path + ".tmp"
            torch.save({"epoch": epoch + 1, "model": trainable_state(model), "opt": opt.state_dict(), "sched": sched.state_dict(),"scaler": scaler.state_dict(), "best": best, "history": hist, "step": step}, tmp)
            os.replace(tmp, last_path)
    if save_path and os.path.exists(save_path):
        model.load_state_dict(torch.load(save_path, map_location=device), strict=False)
        print(f"restored best epoch (dev F1@7 {best:.4f})")
    return hist

@torch.no_grad()
def predict_split(model, split, cfg, D, device, lex_w=0.0, k=7):
    q = encode_docs(model, split["queries"], cfg, D, device, "queries")
    p = encode_docs(model, split["pool"], cfg, D, device, "candidates")
    S = (q @ p.T).numpy()
    if lex_w > 0:
        S = zrow(S) + lex_w * zrow(lexical_scores(split, D))
    top = rank_top(S, split, k)
    return {qd["id"]: [split["pool"][j]["id"] for j in top[i]] for i, qd in enumerate(split["queries"])}

def write_pcr_submission(preds, split, out_dir="/kaggle/working", zip_name="submission_pcr.zip", k=7):
    pool_ids = {p["id"] for p in split["pool"]}
   
    path = os.path.join(out_dir, "prediction.json")
    json.dump(preds, open(path, "w", encoding="utf-8"))
    zp = os.path.join(out_dir, zip_name)
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(path, arcname="prediction.json")
    print(f"wrote {zp}: {len(preds)} queries x {k} candidates")

In [ ]:
CFG_PCR = dict(epochs=5, batch_queries=8, n_hard=4, lr=2e-5, weight_decay=0.01, warmup_ratio=0.1, max_grad_norm=1.0,fp16=True, tau=0.05, pool="mean", window_mode="headtail", train_tokens=1024, eval_tokens=1024,eval_windows=3, eval_batch=16, seed=0)

In [ ]:
CFG_PCR_FULL = (CFG_PCR)          

def run_full_pcr(build_fn, ckpt, D, device, tag="pcr_full", cfg=None, head_path="lm_head", resume=False):
    cfg = cfg or CFG_PCR_FULL
    model = to_encoder(load_phase1(build_fn, ckpt), head_path)
    for p in model.parameters():
        p.requires_grad = True
    count_params(model)
    if "hard_neg" not in D["train"]:
        mine_hard_negatives(D["train"], D, top=50,skip=5)
    hist = train_pcr(model, D["train"], D["dev"], D, cfg, device, save_path=f"/kaggle/working/{tag}_best.pt", resume=resume)
    return model, hist

In [ ]:
class LoRMALinear(nn.Module):
    def __init__(self, base, r=16, alpha=16):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        for p in self.base.parameters():
            p.requires_grad = False

    def forward(self, x):
        z = F.linear(x, self.base.weight)
        y = z + self.scale * F.linear(F.linear(z, self.A), self.B)
        return y + self.base.bias.to(y.dtype) if self.base.bias is not None else y

    @torch.no_grad()
    def to_linear(self):
        W0 = self.base.weight
        W = W0 + self.scale * (self.B @ (self.A @ W0))
        lin = nn.Linear(self.base.in_features, self.base.out_features, bias=self.base.bias is not None).to(W.device, W.dtype)
        lin.weight.copy_(W)
        if self.base.bias is not None:
            lin.bias.copy_(self.base.bias)
        return lin

LORMA_TARGETS = ("q_proj", "k_proj", "v_proj", "c_proj", "c_fc")

def inject_lorma(root, targets=LORMA_TARGETS, r=16, alpha=16):
    names = [n for n, m in root.named_modules() if isinstance(m, nn.Linear) and any(n == t or n.endswith("." + t) for t in targets)]
    for n in names:
        pp, _, ch = n.rpartition(".")
        parent = root.get_submodule(pp) if pp else root
        setattr(parent, ch, LoRMALinear(getattr(parent, ch), r, alpha))
    print(f"LoRMA r={r}, alpha={alpha}: wrapped {len(names)} layers")
    return names

def freeze_for_lorma(model):
    for p in model.parameters():
        p.requires_grad = False
    for m in model.modules():
        if isinstance(m, LoRMALinear):
            m.A.requires_grad = True; m.B.requires_grad = True

@torch.no_grad()
def merge_lorma(root):
    for _, m in list(root.named_modules()):
        for cn, c in list(m.named_children()):
            if isinstance(c, LoRMALinear):
                setattr(m, cn, c.to_linear())

CFG_PCR_LORMA = dict(CFG_PCR)         

def run_lorma_pcr(build_fn, ckpt, D, device, tag="pcr_lorma", r=16, alpha=16, targets=LORMA_TARGETS,
                  cfg=None, head_path="lm_head", resume=False):
    cfg = cfg or CFG_PCR_LORMA
    model = to_encoder(load_phase1(build_fn, ckpt), head_path)
    inject_lorma(model, targets, r, alpha)
    freeze_for_lorma(model)
    count_params(model)
    if "hard_neg" not in D["train"]:
        mine_hard_negatives(D["train"], D, top=50,skip=5)
    hist = train_pcr(model, D["train"], D["dev"], D, cfg, device, save_path=f"/kaggle/working/{tag}_best.pt", resume=resume)
    return model, hist

In [ ]:
NF4_CODE = torch.tensor([
    -1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635, -0.18477343022823334,
    -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141010284, 0.24611230194568634, 0.33791524171829224,
    0.44070982933044434, 0.5626170039176941, 0.7229568362236023, 1.0])

@torch.no_grad()
def quantize_nf4(W, block=64, dq_block=256, chunk=1 << 20):
    shape, n = W.shape, W.numel()
    flat = W.detach().float().flatten().cpu()
    pad = (-n) % block
    if pad:
        flat = torch.cat([flat, flat.new_zeros(pad)])
    blocks = flat.view(-1, block)
    absmax = blocks.abs().amax(1).clamp_min(1e-8)
    normed = blocks / absmax[:, None]
    idx = torch.empty(blocks.shape, dtype=torch.uint8)
    rows = max(1, chunk // block)
    for s in range(0, blocks.size(0), rows):
        idx[s:s + rows] = (normed[s:s + rows, :, None] - NF4_CODE).abs().argmin(-1).to(torch.uint8)
    idx = idx.flatten()
    if idx.numel() % 2:
        idx = torch.cat([idx, idx.new_zeros(1)])
    packed = (idx[0::2] << 4) | idx[1::2]
    offset = absmax.mean()
    a = absmax - offset
    apad = (-a.numel()) % dq_block
    if apad:
        a = torch.cat([a, a.new_zeros(apad)])
    a = a.view(-1, dq_block)
    scale2 = (a.abs().amax(1) / 127).clamp_min(1e-12)
    q_absmax = torch.round(a / scale2[:, None]).clamp(-127, 127).to(torch.int8)
    return {"packed": packed, "q_absmax": q_absmax, "scale2": scale2, "offset": offset.reshape(1),"shape": tuple(shape), "numel": n, "block": block, "n_blocks": blocks.size(0)}

def dequantize_nf4(packed, q_absmax, scale2, offset, code, shape, numel, block, n_blocks, dtype):
    absmax = (q_absmax.float() * scale2[:, None]).flatten()[:n_blocks] + offset
    idx = torch.stack([packed >> 4, packed & 15], dim=-1).flatten()[:n_blocks * block].long()
    w = code[idx].view(n_blocks, block) * absmax[:, None]
    return w.flatten()[:numel].view(shape).to(dtype)

class _NF4Matmul(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, qmod):
        ctx.qmod, ctx.x_dtype = qmod, x.dtype
        return F.linear(x, qmod.dequant(x.dtype))

    @staticmethod
    def backward(ctx, g):
        return (g @ ctx.qmod.dequant(g.dtype)).to(ctx.x_dtype), None

class QLoRALinear(nn.Module):
    def __init__(self, base, r=16, alpha=32, dropout=0.0, block=64, dq_block=256):
        super().__init__()
        q = quantize_nf4(base.weight, block, dq_block)
        self.in_features, self.out_features = base.in_features, base.out_features
        self.meta = {k: q[k] for k in ("shape", "numel", "block", "n_blocks")}
        for k in ("packed", "q_absmax", "scale2", "offset"):
            self.register_buffer(k, q[k])
        self.register_buffer("code", NF4_CODE.clone(), persistent=False)
        self.register_buffer("bias", base.bias.detach().clone() if base.bias is not None else None)
        self.r, self.scale = r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def dequant(self, dtype):
        return dequantize_nf4(self.packed, self.q_absmax, self.scale2, self.offset, self.code, dtype=dtype, **self.meta)

    def forward(self, x):
        y = _NF4Matmul.apply(x, self)
        if self.bias is not None:
            y = y + self.bias.to(y.dtype)
        return y + self.scale * F.linear(F.linear(self.drop(x), self.A), self.B)

    @torch.no_grad()
    def to_linear(self, dtype=torch.float32):
        W = self.dequant(torch.float32) + self.scale * (self.B.float() @ self.A.float())
        lin = nn.Linear(self.in_features, self.out_features, bias=self.bias is not None).to(W.device, dtype)
        lin.weight.copy_(W)
        if self.bias is not None:
            lin.bias.copy_(self.bias)
        return lin

QLORA_TARGETS = ("q_proj", "k_proj", "v_proj", "c_proj", "c_fc")

def quantize_linears(root, targets=QLORA_TARGETS, r=16, alpha=32, dropout=0.0, block=64):
    names = [n for n, m in root.named_modules() if isinstance(m, nn.Linear) and any(n == t or n.endswith("." + t) for t in targets)]
    for n in names:
        pp, _, ch = n.rpartition(".")
        parent = root.get_submodule(pp) if pp else root
        setattr(parent, ch, QLoRALinear(getattr(parent, ch), r, alpha, dropout, block))
    print(f"QLoRA (NF4 block={block}, double-quant) r={r}, alpha={alpha}: quantized {len(names)} layers")
    return names

def freeze_for_qlora(model):
    for p in model.parameters():
        p.requires_grad = False
    for m in model.modules():
        if isinstance(m, QLoRALinear):
            m.A.requires_grad = True; m.B.requires_grad = True

@torch.no_grad()
def merge_qlora(root, dtype=torch.float32):
    for _, m in list(root.named_modules()):
        for cn, c in list(m.named_children()):
            if isinstance(c, QLoRALinear):
                setattr(m, cn, c.to_linear(dtype))

def quant_memory_report(model):
    q = sum(b.numel() * b.element_size() for m in model.modules() if isinstance(m, QLoRALinear) for b in (m.packed, m.q_absmax, m.scale2, m.offset))
    other = sum(p.numel() * p.element_size() for p in model.parameters())
    print(f"4-bit base weights {q/2**20:.1f} MiB | remaining fp params (embeddings, norms, A/B) {other/2**20:.1f} MiB")


def run_qlora_pcr(build_fn, ckpt, D, device, tag="pcr_qlora", r=16, alpha=32, targets=QLORA_TARGETS, dropout=0.05,block=64, cfg=None, head_path="lm_head", resume=False):
    model = to_encoder(load_phase1(build_fn, ckpt), head_path)
    quantize_linears(model, targets, r, alpha, dropout, block)
    freeze_for_qlora(model)
    count_params(model)
    model.to(device)
    quant_memory_report(model)
    if "hard_neg" not in D["train"]:
        mine_hard_negatives(D["train"], D, top=50,skip=5)
    hist = train_pcr(model, D["train"], D["dev"], D, cfg, device, save_path=f"/kaggle/working/{tag}_best.pt", resume=resume)
    return model, hist

In [ ]:
def make_pruned_student(build_fn, ckpt, keep=(0, 2, 4, 6, 8), head_path="lm_head", layers_path="blocks"):
    m = load_phase1(build_fn, ckpt)
    parent = m.get_submodule(layers_path.rpartition(".")[0]) if "." in layers_path else m
    name = layers_path.rpartition(".")[2]
    blocks = getattr(parent, name)
    setattr(parent, name, nn.ModuleList([blocks[i] for i in keep]))
    return to_encoder(m, head_path)

def load_teacher_pcr(build_fn, ckpt, best_path, device, head_path="lm_head", adapter=None):
    t = to_encoder(load_phase1(build_fn, ckpt), head_path)
    if adapter is not None:
        adapter(t)
    t.load_state_dict(torch.load(best_path, map_location="cpu"), strict=False)
    if adapter is not None:
        merge_lorma(t)
    t.to(device).eval()
    for p in t.parameters():
        p.requires_grad = False
    return t

class KDEncoder(nn.Module):
    def __init__(self, student, teacher, pad_id, pool, B, tau_t=0.05, tau_s=0.05):
        super().__init__()
        self.student, self.teacher = student, teacher
        for p in teacher.parameters():
            p.requires_grad = False
        self.pad_id, self.pool, self.B, self.tau_t, self.tau_s, self.kd = pad_id, pool, B, tau_t, tau_s, None

    def train(self, mode=True):
        super().train(mode); self.teacher.eval(); return self

    def forward(self, ids):
        h = get_logits(self.student(ids))
        if self.training:
            lens = (ids != self.pad_id).sum(1)                
            s = pool_hidden(h, lens, self.pool)
            with torch.no_grad():
                t = pool_hidden(get_logits(self.teacher(ids)), lens, self.pool)
            B = self.B
            ss, ts = (s[:B] @ s[B:].T).float(), (t[:B] @ t[B:].T).float()
            self.kd = F.kl_div(F.log_softmax(ss / self.tau_s, -1), F.softmax(ts / self.tau_t, -1), reduction="batchmean")
        return h

def run_kd_pcr(build_fn, ckpt, teacher, D, device, tag="pcr_kd", keep=(0, 2, 4, 6, 8), kd_weight=1.0,cfg=None, head_path="lm_head", resume=False):
    student = make_pruned_student(build_fn, ckpt, keep, head_path)
    for p in student.parameters():
        p.requires_grad = True
    count_params(student)
    wrap = KDEncoder(student, teacher, D["pad_id"], cfg["pool"], cfg["batch_queries"]).to(device)
    if "hard_neg" not in D["train"]:
        mine_hard_negatives(D["train"], D)
    hist = train_pcr(wrap, D["train"], D["dev"], D, cfg, device, save_path=f"/kaggle/working/{tag}_best.pt",
                     extra_loss=lambda m: kd_weight * m.kd, resume=resume)
    return wrap.student, hist

In [ ]:
P7 = "Phase_7_PCR"
hf_dataset_repo = "Exploration-Lab/CS-781-Capstone"
p7 = download_pcr_files(hf_dataset_repo, P7, HF_TOKEN)           

for s, (qf, cf) in p7.items():
    for nm, f in (("queries", qf), ("candidates", cf)):
        raw = json.load(open(f, encoding="utf-8"))
        rec = raw[0] if isinstance(raw, list) else next(iter(raw.items()))
        print(s, nm, type(raw).__name__, len(raw), "| first record:", str(rec)[:300])

D = setup_pcr(p7, "/kaggle/working/legal_bpe_tokenizer.pkl", cache_prefix="/kaggle/working/pcr_tok")

In [ ]:
paths = download_pcr_files(hf_dataset_repo, P7, HF_TOKEN, include_test=True)
add_split(D, paths, "test", cache_prefix="/kaggle/working/pcr_tok")

In [ ]:
VOCAB_SIZE=D['vocab_size']
PAD_ID=D['pad_id']
cfg = CFG_PCR

model, hist = run_full_pcr(build_model, MODEL_PT, D, device, tag="pcr_full_v6")
print([(h["epoch"], round(h["val"], 4)) for h in hist])
lex_dev = lexical_scores(D["dev"], D)
m, dense_dev = evaluate_pcr(model, D["dev"], cfg, D, device)
w = tune_hybrid(D["dev"], dense_dev, lex_dev)



preds = predict_split(model, D["test"], cfg, D, device, lex_w=w)      
write_pcr_submission(preds, D["test"], "/kaggle/working", "submission_pcr_full_v6.zip")

In [ ]:
model_lorma, hist = run_lorma_pcr(build_model, MODEL_PT, D, device, tag="pcr_lorma_v1", r=16, alpha=16)
print([(h["epoch"], round(h["val"], 4)) for h in hist])


m, dense_dev = evaluate_pcr(model_lorma, D["dev"], cfg, D, device)
w = tune_hybrid(D["dev"], dense_dev, lexical_scores(D["dev"], D))

preds = predict_split(model_lorma, D["test"], cfg, D, device, lex_w=w)    
write_pcr_submission(preds, D["test"], "/kaggle/working", "submission_pcr_lorma.zip")

In [ ]:
model_q, hist = run_qlora_pcr(build_model, MODEL_PT, D, device, tag="pcr_qlora_v1", r=16, alpha=32,cfg=cfg)
print([(h["epoch"], round(h["val"], 4)) for h in hist])


m, dense_dev = evaluate_pcr(model_q, D["dev"], cfg, D, device)
w = tune_hybrid(D["dev"], dense_dev, lexical_scores(D["dev"], D))

preds = predict_split(model_q, D["test"], cfg, D, device, lex_w=w)     
write_pcr_submission(preds, D["test"],"/kaggle/working","submission_pcr_qlora.zip")

In [ ]:
teacher = load_teacher_pcr(build_model, MODEL_PT, "/kaggle/working/pcr_full_v1_best.pt", device)              
student, hist = run_kd_pcr(build_model, MODEL_PT, teacher, D, device, tag="pcr_kd_v1", keep=(0,2,4,6,8), kd_weight=1.0,cfg=cfg)
print([(h["epoch"], round(h["val"], 4)) for h in hist])

m, dense_dev = evaluate_pcr(student, D["dev"], cfg, D, device)
w = tune_hybrid(D["dev"], dense_dev, lexical_scores(D["dev"], D))

preds = predict_split(student, D["test"], cfg, D, device, lex_w=w)
write_pcr_submission(preds, D["test"], "/kaggle/working", "submission_pcr_kd.zip")